In [1]:
# ============================================================
# STEP 1 — DATA IMPORT AND VARIABLE PREPARATION
# ============================================================
"""
Input  : raw CSV, ';'-separated, comma decimal
Output : outputs/data_prepared.csv, reused as-is by every subsequent
         step (02_pearson_correlation.py, 03_pca_weights.py, etc.)
"""

import sys
from pathlib import Path

import pandas as pd

# ============================================================
# CONFIGURATION — the only block to edit when switching datasets
# ============================================================

INPUT_FILE = Path("SFI_CALCULATION.csv")
PREPARED_DATA_FILE = Path("outputs/data_prepared.csv")

# Unique observation identifier
ID_COLUMN = "ID_LABO"

# Raw column -> final column name, converted to numeric (decimal = point)
NUMERIC_COLUMNS = {
    "PH": "pH",
    "K": "K",
    "P": "P",
    "CEC": "CEC",
    "ARGILE": "clay",
    "LIMON": "silt",
    "SABLE": "sand",
    "N": "N",
    "C": "C",
}

# Depth is coded numerically (1-5); same depth classes across all sites
DEPTH_ORDER = [1, 2, 3, 4, 5]
DEPTH_TO_CM = {1: 10, 2: 20, 3: 30, 4: 60, 5: 90}

# Texture -> abbreviated code (USDA-style), used later for the
# Table 4 (Sulaeman et al., 2021) texture-based scoring in step 4.
# NOTE: "SILT" and "SILT LOAM" are required for the VL and M classes
# of the texture scoring — do not remove them.
TEXTURE_CODE = {
    "CLAY": "C", "CLAY LOAM": "CL", "LOAM": "L", "SAND": "S",
    "SANDY CLAY": "SC", "SANDY CLAY LOAM": "SCL", "SANDY LOAM": "SL",
    "LOAMY SAND": "LS", "SILTY CLAY": "SiC", "SILTY CLAY LOAM": "SiCL",
    "SILT": "Si", "SILT LOAM": "SiL",
}


# ============================================================
# FUNCTIONS
# ============================================================

def load_raw_data(path: Path) -> pd.DataFrame:
    """Load the raw CSV with automatic field-separator detection."""
    if not path.exists():
        sys.exit(f"Error: file not found -> {path}")

    df = pd.read_csv(path, sep=None, engine="python", decimal=".")

    if df.shape[1] == 1:
        sys.exit(
            "Error: only one column detected — the field separator was "
            "likely not identified correctly. Check the file or set "
            "sep=... explicitly."
        )
    return df


def convert_numeric_columns(df: pd.DataFrame, mapping: dict) -> pd.DataFrame:
    """Convert comma-decimal text columns to float and rename them."""
    for raw_col, final_col in mapping.items():
        df[final_col] = pd.to_numeric(
            df[raw_col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )
    return df


def add_categorical_factors(df: pd.DataFrame) -> pd.DataFrame:
    """Define categorical factors: Site, LandUse, Texture."""
    df["Site"] = df["Site"].astype("category")
    df["LandUse"] = df["Landuse"].astype("category")
    df["Texture"] = df["TEXTURE"].astype("category")
    return df


def add_depth_variables(df: pd.DataFrame, order: list, cm_mapping: dict) -> pd.DataFrame:
    """Add depth as an ordered factor (Depth_class) and its value in cm (Depth_cm)."""
    df["Depth_class"] = pd.Categorical(df["Depth"], categories=order, ordered=True)
    df["Depth_cm"] = df["Depth"].map(cm_mapping)
    return df


def add_texture_code(df: pd.DataFrame, code_mapping: dict) -> pd.DataFrame:
    """Add a 'Texture_code' column (short abbreviation, e.g. SiCL, CL, S)."""
    df["Texture_code"] = df["TEXTURE"].str.upper().map(code_mapping)
    unmatched = df.loc[df["Texture_code"].isna() & df["TEXTURE"].notna(), "TEXTURE"].unique()
    if len(unmatched) > 0:
        print(f"WARNING: unrecognized texture classes in TEXTURE_CODE: {list(unmatched)}")
    return df


def check_id_uniqueness(df: pd.DataFrame, id_col: str) -> None:
    """Check that the identifier column has no missing values or duplicates."""
    if id_col not in df.columns:
        print(f"WARNING: identifier column '{id_col}' is missing from the file.")
        return
    n_missing = df[id_col].isna().sum()
    n_duplicates = df[id_col].duplicated().sum()
    print(f"=== Identifier check ('{id_col}') ===")
    print(f"Observations: {len(df)} | Missing: {n_missing} | Duplicates: {n_duplicates}")
    if n_missing or n_duplicates:
        print("WARNING: check the identifiers before proceeding with the analysis.")


def check_data_quality(df: pd.DataFrame, columns: list) -> None:
    """Report missing values per column and flag any fully-empty column."""
    n_na = df[columns].isna().sum()
    print("=== Missing values per column (after conversion) ===")
    print(n_na)
    fully_empty = n_na[n_na == len(df)].index.tolist()
    if fully_empty:
        print(f"\nWARNING: fully empty column(s) -> {fully_empty}. "
              f"Check the corresponding raw column name.")


def save_prepared_data(df: pd.DataFrame, path: Path) -> None:
    """Save the prepared DataFrame, to be reloaded explicitly by later steps."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"\nSaved prepared data -> {path}")


def prepare_data(input_file: Path = INPUT_FILE, output_file: Path = PREPARED_DATA_FILE) -> pd.DataFrame:
    """Full pipeline: load + prepare variables."""
    df = load_raw_data(input_file)
    df = convert_numeric_columns(df, NUMERIC_COLUMNS)
    df = add_categorical_factors(df)
    df = add_depth_variables(df, DEPTH_ORDER, DEPTH_TO_CM)
    df = add_texture_code(df, TEXTURE_CODE)

    check_id_uniqueness(df, ID_COLUMN)
    check_data_quality(df, list(NUMERIC_COLUMNS.values()))

    print("\n=== Prepared dataset preview ===")
    print(df.info())
    print(df.head())

    save_prepared_data(df, output_file)
    return df


# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    data = prepare_data()


=== Identifier check ('ID_LABO') ===
Observations: 1529 | Missing: 0 | Duplicates: 0
=== Missing values per column (after conversion) ===
pH      0
K       0
P       0
CEC     0
clay    0
silt    0
sand    0
N       0
C       0
dtype: int64

=== Prepared dataset preview ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1529 entries, 0 to 1528
Data columns (total 23 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   ID_LABO       1529 non-null   int64   
 1   Site          1529 non-null   category
 2   Depth         1529 non-null   int64   
 3   Landuse       1529 non-null   object  
 4   PH            1529 non-null   float64 
 5   P             1529 non-null   float64 
 6   K             1529 non-null   float64 
 7   N             1529 non-null   float64 
 8   C             1529 non-null   float64 
 9   CEC           1529 non-null   float64 
 10  SABLE         1529 non-null   float64 
 11  ARGILE        1529 non-null   float64 
 1

In [2]:
# ============================================================
# STEP 2 — PEARSON CORRELATION MATRIX + P-VALUES
# ============================================================
"""

Input  : outputs/data_prepared.csv, produced by 01_data_preparation.py
Output : - outputs/correlation_heatmap.png
         - outputs/pearson_correlation_matrix.csv (full r matrix)
         - outputs/pearson_pvalue_matrix.csv      (full p matrix)
         - outputs/pearson_significant_pairs.csv
         - outputs/pearson_strong_pairs.csv (|r| > 0.75, MSFI threshold)
"""

from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr

# ============================================================
# CONFIGURATION
# ============================================================

# Variables included in the correlation analysis, using the same names
# produced by 01_data_preparation.py (see NUMERIC_COLUMNS)
CORR_VARIABLES = ["N", "P", "C", "CEC", "pH", "K", "clay", "silt", "sand"]

ALPHA = 0.05

# |r| threshold used for MSFI collinearity screening (variables above
# this threshold are candidates for exclusion — see note above for the
# clay/sand case, which additionally rests on expert judgement)
MSFI_CORRELATION_THRESHOLD = 0.75

OUTPUT_DIR = Path("outputs")
PREPARED_DATA_FILE = OUTPUT_DIR / "data_prepared.csv"


# ============================================================
# FUNCTIONS
# ============================================================

def load_prepared_data(path: Path) -> pd.DataFrame:
    """Load the data prepared by 01_data_preparation.py."""
    if not path.exists():
        raise FileNotFoundError(
            f"File not found -> {path}. Did you run 01_data_preparation.py first?"
        )
    return pd.read_csv(path)


def compute_pearson_matrices(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Equivalent of Hmisc::rcorr: computes the Pearson correlation matrix
    (r) and the associated p-value matrix for every pair of columns in
    `df`. Each pair is computed once (the matrix is symmetric) and then
    mirrored on both sides.
    """
    cols = df.columns
    r_matrix = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)
    p_matrix = pd.DataFrame(np.zeros((len(cols), len(cols))), index=cols, columns=cols)

    for col_a, col_b in combinations(cols, 2):
        valid = df[[col_a, col_b]].dropna()
        r, p = pearsonr(valid[col_a], valid[col_b])
        r_matrix.loc[col_a, col_b] = r_matrix.loc[col_b, col_a] = r
        p_matrix.loc[col_a, col_b] = p_matrix.loc[col_b, col_a] = p

    return r_matrix, p_matrix


def plot_correlation_heatmap(cor_matrix: pd.DataFrame, output_path: Path) -> None:
    """Generate and save the correlation heatmap (equivalent of corrplot)."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8, 7))
    sns.heatmap(
        cor_matrix, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
        square=True, cbar_kws={"shrink": 0.8},
    )
    plt.title("Pearson correlation matrix")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def extract_pair_results(cor_matrix: pd.DataFrame, p_matrix: pd.DataFrame) -> pd.DataFrame:
    """Build a long table (one row per variable pair, no Var1/Var2 <-> Var2/Var1 duplicate)."""
    cols = cor_matrix.columns
    rows = [
        {
            "Var1": col_a, "Var2": col_b,
            "r": cor_matrix.loc[col_a, col_b],
            "p": p_matrix.loc[col_a, col_b],
        }
        for col_a, col_b in combinations(cols, 2)
    ]
    return pd.DataFrame(rows).sort_values("r", key=abs, ascending=False).reset_index(drop=True)


def select_significant_pairs(pairs: pd.DataFrame, alpha: float) -> pd.DataFrame:
    """Filter statistically significant pairs (p < alpha)."""
    return pairs[pairs["p"] < alpha].reset_index(drop=True)


def select_msfi_candidates(pairs: pd.DataFrame, r_threshold: float) -> pd.DataFrame:
    """
    Filter pairs whose |r| exceeds the collinearity threshold retained
    for MSFI selection. These are the pairs where only one of the two
    variables should be kept in the analysis.
    """
    return pairs[pairs["r"].abs() > r_threshold].reset_index(drop=True)


def run_pearson_analysis(
    data_file: Path = PREPARED_DATA_FILE,
    variables: list = CORR_VARIABLES,
    alpha: float = ALPHA,
    msfi_threshold: float = MSFI_CORRELATION_THRESHOLD,
    output_dir: Path = OUTPUT_DIR,
) -> dict:
    """Full pipeline for step 2. Loads the data, analyses, exports, and returns the results."""
    data = load_prepared_data(data_file)

    missing = [v for v in variables if v not in data.columns]
    if missing:
        raise KeyError(
            f"Columns missing from `data`: {missing}. "
            f"Check that they were produced by 01_data_preparation.py."
        )

    sub = data[variables]
    cor_matrix, p_matrix = compute_pearson_matrices(sub)

    output_dir.mkdir(parents=True, exist_ok=True)
    cor_matrix.to_csv(output_dir / "pearson_correlation_matrix.csv")
    p_matrix.to_csv(output_dir / "pearson_pvalue_matrix.csv")
    plot_correlation_heatmap(cor_matrix, output_dir / "correlation_heatmap.png")

    all_pairs = extract_pair_results(cor_matrix, p_matrix)
    sig_pairs = select_significant_pairs(all_pairs, alpha)
    msfi_pairs = select_msfi_candidates(sig_pairs, msfi_threshold)

    sig_pairs.to_csv(output_dir / "pearson_significant_pairs.csv", index=False)
    msfi_pairs.to_csv(output_dir / "pearson_strong_pairs.csv", index=False)

    print(f"=== Significant correlations (p < {alpha}): {len(sig_pairs)} pairs ===")
    print(sig_pairs)
    print(f"\n=== Pairs with |r| > {msfi_threshold} — statistical basis for MSFI collinearity screening ===")
    print(msfi_pairs)

    return {
        "cor_matrix": cor_matrix,
        "p_matrix": p_matrix,
        "all_pairs": all_pairs,
        "sig_pairs": sig_pairs,
        "msfi_pairs": msfi_pairs,
    }


# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    # Explicitly loads outputs/data_prepared.csv (produced by
    # 01_data_preparation.py) — no dependency on an in-memory session.
    results = run_pearson_analysis()


=== Significant correlations (p < 0.05): 33 pairs ===
    Var1  Var2         r              p
0      N     C  0.947253   0.000000e+00
1   clay  sand -0.936202   0.000000e+00
2   silt  sand -0.850559   0.000000e+00
3   clay  silt  0.611467  1.840399e-157
4    CEC  clay  0.579080  1.271891e-137
5    CEC  sand -0.567021  8.663235e-131
6      N     K  0.423058   1.959044e-67
7    CEC  silt  0.410114   4.291501e-63
8      N     P  0.404910   2.111542e-61
9      P     C  0.387951   4.298866e-56
10     C     K  0.360323   4.345446e-48
11     P  silt -0.343055   1.800729e-43
12     C    pH -0.313036   4.121036e-36
13     C   CEC  0.285051   5.644633e-30
14     P  sand  0.284792   6.385127e-30
15     N    pH -0.276158   3.643093e-28
16   CEC     K  0.273724   1.110217e-27
17     N   CEC  0.273374   1.301570e-27
18    pH  silt  0.257272   1.543384e-24
19     P    pH -0.224244   7.019750e-19
20    pH  sand -0.209269   1.362911e-16
21     K  clay  0.207300   2.647261e-16
22     P  clay -0.199240  

In [14]:
# ============================================================
# STEP 3 — PRINCIPAL COMPONENT ANALYSIS (PCA)
# + Wj WEIGHT CALCULATION PER EQUATION (1)
#   Wj = |xjk| . Wk
#   xjk : loading of variable j on ITS corresponding component k
#         (the retained component where |loading| is maximal)
#   Wk  : eigenvalue of that component k
# ============================================================
"""

Input  : outputs/data_prepared.csv, produced by 01_data_preparation.py
Output : - outputs/pca_eigenvalues.csv
         - outputs/pca_variable_coordinates.csv
         - outputs/pca_variable_contributions.csv
         - outputs/pca_weights_Wj.csv (used in step 5 for the SFI)
         - outputs/pca_correlation_circle.png (variable correlation circle, PC1 x PC2)
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("outputs")
PREPARED_DATA_FILE = OUTPUT_DIR / "data_prepared.csv"

# The 6 MSFI indicators retained in the methodology (after the
# collinearity screening in step 2): pH, CEC, clay (continuous proxy for
# texture, retained over sand/silt), K, P, C content.
PCA_VARIABLES = ["pH", "CEC", "clay", "K", "P", "C"]

# Kaiser criterion: components retained if eigenvalue > this threshold
KAISER_THRESHOLD = 1.0

# Normalize Wj weights to sum = 1 (see docstring note above)
NORMALIZE_WEIGHTS = True

# Mapping to the score column names used in steps 4/5 ("clay" ->
# "texture_score" because the final score for this indicator is derived
# from the Table 4 texture class, not directly from % clay)
VAR_TO_SCORE_NAME = {
    "pH": "pH_score",
    "CEC": "CEC_score",
    "clay": "texture_score",
    "K": "K_score",
    "P": "P_score",
    "C": "C_score",
}


# ============================================================
# FUNCTIONS
# ============================================================

def load_prepared_data(path: Path) -> pd.DataFrame:
    """Load the data prepared by 01_data_preparation.py."""
    if not path.exists():
        raise FileNotFoundError(
            f"File not found -> {path}. Did you run 01_data_preparation.py first?"
        )
    return pd.read_csv(path)


def select_pca_data(df: pd.DataFrame, variables: list) -> pd.DataFrame:
    """Select the MSFI variables and drop incomplete observations (PCA needs complete cases)."""
    missing_cols = [v for v in variables if v not in df.columns]
    if missing_cols:
        raise KeyError(f"MSFI columns missing from `data`: {missing_cols}")

    pca_data = df[variables].dropna()
    n_dropped = len(df) - len(pca_data)
    pct_dropped = 100 * n_dropped / len(df) if len(df) else 0
    print(f"Complete observations for PCA: {len(pca_data)}/{len(df)} "
          f"({n_dropped} excluded, {pct_dropped:.1f}%)")
    if pct_dropped > 10:
        print("WARNING: more than 10% of observations are excluded due to missing values.")
    return pca_data


def standardize(pca_data: pd.DataFrame) -> np.ndarray:
    """Standardization (equivalent of scale.unit=TRUE in FactoMineR::PCA)."""
    return StandardScaler().fit_transform(pca_data)


def run_pca(X_scaled: np.ndarray, variables: list) -> tuple[PCA, pd.DataFrame]:
    """Fit the PCA and build the eigenvalue table (res.pca$eig)."""
    pca = PCA()
    pca.fit(X_scaled)

    eigenvalues = pca.explained_variance_
    variance_pct = pca.explained_variance_ratio_ * 100
    cum_variance_pct = np.cumsum(variance_pct)

    eig_table = pd.DataFrame(
        {
            "eigenvalue": eigenvalues,
            "percentage_of_variance": variance_pct,
            "cumulative_percentage": cum_variance_pct,
        },
        index=[f"Dim.{i + 1}" for i in range(len(eigenvalues))],
    )
    return pca, eig_table


def compute_variable_coordinates(pca: PCA, eig_table: pd.DataFrame, variables: list) -> pd.DataFrame:
    """Variable coordinates on the components (res.pca$var$coord): loading x sqrt(eigenvalue)."""
    loadings = pca.components_.T
    var_coord = loadings * np.sqrt(eig_table["eigenvalue"].values)
    return pd.DataFrame(var_coord, index=variables, columns=eig_table.index)


def compute_contributions(var_coord_df: pd.DataFrame) -> pd.DataFrame:
    """Variable contributions to each component (res.pca$var$contrib, in %)."""
    contrib = var_coord_df ** 2
    return contrib.div(contrib.sum(axis=0), axis=1) * 100


def select_retained_components(eig_table: pd.DataFrame, threshold: float) -> list:
    """Kaiser criterion: components with eigenvalue > threshold."""
    retained = list(eig_table.index[eig_table["eigenvalue"] > threshold])
    if not retained:
        raise ValueError(f"No component with eigenvalue > {threshold}: check the input data.")
    print(f"Retained components (eigenvalue > {threshold}): {retained}")
    return retained


def compute_wj_weights(
    var_coord_df: pd.DataFrame,
    eig_table: pd.DataFrame,
    variables: list,
    retained_dims: list,
    normalize: bool = True,
) -> pd.DataFrame:
    """
    Compute the weight Wj of each indicator per equation (1):
    Wj = |xjk| . Wk, where xjk is the loading of variable j on its
    "corresponding" component k (the retained component where |loading|
    is maximal), and Wk the eigenvalue of that component.
    """
    loadings_retained = var_coord_df[retained_dims]

    assigned_component = {}
    raw_weights = {}
    for var in variables:
        abs_loadings = loadings_retained.loc[var].abs()
        best_dim = abs_loadings.idxmax()
        xjk = loadings_retained.loc[var, best_dim]
        Wk = eig_table.loc[best_dim, "eigenvalue"]
        raw_weights[var] = abs(xjk) * Wk
        assigned_component[var] = best_dim

    raw_weights = pd.Series(raw_weights, name="Wj_raw")
    weights = raw_weights / raw_weights.sum() if normalize else raw_weights.copy()
    weights.name = "Wj_normalized" if normalize else "Wj_raw"

    weights_table = pd.DataFrame({
        "matching_component": pd.Series(assigned_component),
        "abs_loading": loadings_retained.abs().max(axis=1),
        "eigenvalue_Wk": pd.Series(
            {v: eig_table.loc[assigned_component[v], "eigenvalue"] for v in variables}
        ),
        "Wj_raw": raw_weights,
        "Wj_normalized": raw_weights / raw_weights.sum(),
    })
    return weights_table


def plot_correlation_circle(var_coord_df: pd.DataFrame, eig_table: pd.DataFrame, output_path: Path) -> None:
    """Plot the PCA variable correlation circle (PC1 x PC2), one arrow per property."""
    pc1_pct = eig_table.loc["Dim.1", "percentage_of_variance"]
    pc2_pct = eig_table.loc["Dim.2", "percentage_of_variance"]

    fig, ax = plt.subplots(figsize=(6, 6))
    circle = plt.Circle((0, 0), 1, fill=False, color="grey", linestyle="--")
    ax.add_patch(circle)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.axvline(0, color="grey", linewidth=0.8)

    for var in var_coord_df.index:
        x, y = var_coord_df.loc[var, "Dim.1"], var_coord_df.loc[var, "Dim.2"]
        ax.annotate(
            "", xy=(x, y), xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.5),
        )
        ax.text(x * 1.1, y * 1.1, var, ha="center", va="center", fontsize=10)

    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.set_xlabel(f"PC1 ({pc1_pct:.1f}%)")
    ax.set_ylabel(f"PC2 ({pc2_pct:.1f}%)")
    ax.set_title("PCA correlation circle — MSFI indicators")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def run_pca_analysis(
    data_file: Path = PREPARED_DATA_FILE,
    variables: list = PCA_VARIABLES,
    kaiser_threshold: float = KAISER_THRESHOLD,
    normalize_weights: bool = NORMALIZE_WEIGHTS,
    var_to_score: dict = VAR_TO_SCORE_NAME,
    output_dir: Path = OUTPUT_DIR,
) -> dict:
    """Full pipeline for step 3. Loads, computes, exports, and returns the results."""
    data = load_prepared_data(data_file)
    pca_data = select_pca_data(data, variables)

    X_scaled = standardize(pca_data)
    pca, eig_table = run_pca(X_scaled, variables)
    var_coord_df = compute_variable_coordinates(pca, eig_table, variables)
    contrib_df = compute_contributions(var_coord_df)
    retained_dims = select_retained_components(eig_table, kaiser_threshold)
    weights_table = compute_wj_weights(
        var_coord_df, eig_table, variables, retained_dims, normalize=normalize_weights
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    eig_table.to_csv(output_dir / "pca_eigenvalues.csv")
    var_coord_df.to_csv(output_dir / "pca_variable_coordinates.csv")
    contrib_df.to_csv(output_dir / "pca_variable_contributions.csv")
    weights_table.to_csv(output_dir / "pca_weights_Wj.csv")
    plot_correlation_circle(var_coord_df, eig_table, output_dir / "pca_correlation_circle.png")

    print("\n=== Eigenvalues (res.pca$eig) ===")
    print(eig_table)
    print("\n=== Variable coordinates (res.pca$var$coord) ===")
    print(var_coord_df)
    print("\n=== Variable contributions (res.pca$var$contrib, %) ===")
    print(contrib_df)
    print("\n=== Wj weights (equation 1) ===")
    print(weights_table)

    weights_scores = weights_table["Wj_normalized"].rename(index=var_to_score)
    print("\n=== Wj weights (indexed on score column names, used in step 5) ===")
    print(weights_scores)

    return {
        "pca": pca,
        "eig_table": eig_table,
        "var_coord": var_coord_df,
        "contrib": contrib_df,
        "retained_dims": retained_dims,
        "weights_table": weights_table,
        "weights_scores": weights_scores,
    }


# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    results = run_pca_analysis()


Complete observations for PCA: 1529/1529 (0 excluded, 0.0%)
Retained components (eigenvalue > 1.0): ['Dim.1', 'Dim.2']

=== Eigenvalues (res.pca$eig) ===
       eigenvalue  percentage_of_variance  cumulative_percentage
Dim.1    1.938504               32.287276              32.287276
Dim.2    1.652169               27.518141              59.805417
Dim.3    0.891788               14.853415              74.658832
Dim.4    0.721663               12.019852              86.678684
Dim.5    0.434309                7.233751              93.912435
Dim.6    0.365493                6.087565             100.000000

=== Variable coordinates (res.pca$var$coord) ===
         Dim.1     Dim.2     Dim.3     Dim.4     Dim.5     Dim.6
pH   -0.042699 -0.672763  0.595890  0.357492  0.232534 -0.096247
CEC   0.787427 -0.295009 -0.224384  0.263095  0.048425  0.414331
clay  0.633787 -0.547382 -0.344730  0.034403 -0.180289 -0.383165
K     0.644458  0.015473  0.576107 -0.432199 -0.249863  0.062951
P     0.244581  

In [2]:
# ============================================================
# STEP 4 — INDICATOR SCORING (Table 4, Sulaeman et al., 2021)
# ============================================================
"""

Input  : outputs/data_prepared.csv, produced by 01_data_preparation.py
         (expected columns: pH, CEC, K, P, C, Texture_code)

Small gaps between classes are assigned to the adjacent higher class, 
following the contiguous-bin convention of pd.cut

Output : outputs/data_scored.csv, reused by 05_sfi_calculation.py

"""

from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("outputs")
PREPARED_DATA_FILE = OUTPUT_DIR / "data_prepared.csv"
SCORED_DATA_FILE = OUTPUT_DIR / "data_scored.csv"

# Number of SFI classes (n in equation 2) -> p = 1/n
N_CLASSES = 5

# --- Scoring thresholds (Table 4, Sulaeman et al., 2021 — corrected) ---
SCORE_BINS = {
    "pH": [-np.inf, 5, 5.4, 5.8, 6, 7, 7.4, 7.8, 8.2, np.inf],
    "CEC": [-np.inf, 5, 16, 24, 40, np.inf],
    "K": [-np.inf, 0.1, 0.3, 0.5, 1, np.inf],
    "P": [-np.inf, 4, 7, 10, 15, np.inf],
    "C": [-np.inf, 1, 2, 3, 5, np.inf],
}

# Per-variable labels: pH needs 9 (bell-shaped, non-monotonic),
# the others need 5 (monotonic increasing).
SCORE_LABELS = {
    "pH": [1, 2, 3, 4, 5, 4, 3, 2, 1],
    "CEC": [1, 2, 3, 4, 5],
    "K": [1, 2, 3, 4, 5],
    "P": [1, 2, 3, 4, 5],
    "C": [1, 2, 3, 4, 5],
}

# Texture classification (Table 4), from Texture_code (step 1)
TEXTURE_CLASS = {
    "S": 1, "Si": 1,
    "LS": 2,
    "SL": 3, "L": 3, "SiL": 3,
    "SiC": 4, "CL": 4, "SCL": 4,
    "SiCL": 5, "SC": 5, "C": 5,
}

# Organic carbon unit conversion: g/kg -> %
CONVERT_C_GKG_TO_PERCENT = True

SCORE_COLUMNS = ["pH_score", "CEC_score", "K_score", "P_score", "C_score", "texture_score"]


# ============================================================
# FUNCTIONS
# ============================================================

def load_prepared_data(path: Path) -> pd.DataFrame:
    """Load the data prepared by 01_data_preparation.py."""
    if not path.exists():
        raise FileNotFoundError(
            f"File not found -> {path}. Did you run 01_data_preparation.py first?"
        )
    return pd.read_csv(path)


def convert_carbon_units(df: pd.DataFrame, enabled: bool) -> pd.DataFrame:
    """Convert C from g/kg to % if enabled=True."""
    if enabled:
        df["C"] = df["C"] / 10
    print("=== C statistics (after optional unit conversion) ===")
    print(df["C"].describe())
    return df


def score_variable(series: pd.Series, bins: list, labels: list) -> pd.Series:
    """Cut a continuous variable into classes 1-5 per Table 4 thresholds.

    ordered=False is required because pH's labels are non-monotonic
    and contain duplicates (1,2,3,4,5,4,3,2,1); pd.cut would otherwise
    raise on the duplicate categories.
    """
    return pd.cut(series, bins=bins, labels=labels, ordered=False).astype(int)


def score_texture(df: pd.DataFrame, texture_class: dict) -> pd.Series:
    """Derive the texture score (1-5) from Texture_code (produced in step 1)."""
    if "Texture_code" not in df.columns:
        raise KeyError(
            "Column 'Texture_code' is missing: check that it was produced "
            "by 01_data_preparation.py."
        )
    scores = df["Texture_code"].map(texture_class)

    unmatched = df.loc[scores.isna() & df["Texture_code"].notna(), "Texture_code"].unique()
    if len(unmatched) > 0:
        print(f"WARNING: unrecognized texture codes in TEXTURE_CLASS: {list(unmatched)}. "
              f"Check TEXTURE_CODE in 01_data_preparation.py.")
    return scores


def apply_all_scores(df: pd.DataFrame, bins_config: dict, labels_config: dict, texture_class: dict) -> pd.DataFrame:
    """Apply Table 4 scoring to all MSFI indicators.

    labels_config is a dict {variable: labels}, since pH needs 9 labels
    while the other indicators need 5 — a single shared label list no
    longer works once pH's bell-shaped curve is included.
    """
    for var, bins in bins_config.items():
        df[f"{var}_score"] = score_variable(df[var], bins, labels_config[var])
    df["texture_score"] = score_texture(df, texture_class)
    return df


def standardize_scores(df: pd.DataFrame, score_cols: list, n_classes: int) -> pd.DataFrame:
    """
    Standardize each score: Sij . p, with p = 1/n_classes (equation 2).
    Applied only once here — do not divide the SFI by n_classes again
    later (see 05_sfi_calculation.py).
    """
    p = 1 / n_classes
    for col in score_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce") * p
    return df


def check_score_quality(df: pd.DataFrame, score_cols: list) -> None:
    """Check for missing values after scoring (consistent with steps 1-3)."""
    n_na = df[score_cols].isna().sum()
    print("\n=== Missing values per score (after standardization) ===")
    print(n_na)
    fully_empty = n_na[n_na == len(df)].index.tolist()
    if fully_empty:
        print(f"\nWARNING: fully missing score(s) -> {fully_empty}.")


def save_scored_data(df: pd.DataFrame, path: Path) -> None:
    """Save the scored DataFrame, reloaded explicitly by 05_sfi_calculation.py."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"\nSaved scored data -> {path}")


def run_scoring(
    data_file: Path = PREPARED_DATA_FILE,
    output_file: Path = SCORED_DATA_FILE,
    bins_config: dict = SCORE_BINS,
    labels: dict = SCORE_LABELS,
    texture_class: dict = TEXTURE_CLASS,
    n_classes: int = N_CLASSES,
    convert_carbon: bool = CONVERT_C_GKG_TO_PERCENT,
) -> pd.DataFrame:
    """Full pipeline for step 4."""
    data = load_prepared_data(data_file)

    required = ["pH", "CEC", "K", "P", "C", "Texture_code"]
    missing = [v for v in required if v not in data.columns]
    if missing:
        raise KeyError(
            f"Required columns missing from `data`: {missing}. "
            f"Check 01_data_preparation.py."
        )

    data = convert_carbon_units(data, convert_carbon)
    data = apply_all_scores(data, bins_config, labels, texture_class)

    print("\n=== Texture class distribution (before standardization) ===")
    print(data["texture_score"].value_counts(dropna=False))

    data = standardize_scores(data, SCORE_COLUMNS, n_classes)
    check_score_quality(data, SCORE_COLUMNS)

    print("\n=== Standardized scores (Sij . p) ===")
    print(data[SCORE_COLUMNS].describe())

    save_scored_data(data, output_file)
    return data


# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    data = run_scoring()

=== C statistics (after optional unit conversion) ===
count    1.529000e+03
mean     1.712114e+00
std      2.073391e+00
min      7.490000e-11
25%      3.906000e-01
50%      9.462820e-01
75%      2.364755e+00
max      1.950594e+01
Name: C, dtype: float64

=== Texture class distribution (before standardization) ===
texture_score
4    597
3    414
1    228
5    154
2    136
Name: count, dtype: int64

=== Missing values per score (after standardization) ===
pH_score         0
CEC_score        0
K_score          0
P_score          0
C_score          0
texture_score    0
dtype: int64

=== Standardized scores (Sij . p) ===
          pH_score    CEC_score      K_score      P_score      C_score  \
count  1529.000000  1529.000000  1529.000000  1529.000000  1529.000000   
mean      0.613080     0.549640     0.281622     0.218967     0.405232   
std       0.230493     0.175163     0.104021     0.080777     0.261337   
min       0.200000     0.200000     0.200000     0.200000     0.200000   
25%   

In [3]:
# ============================================================
# STEP 5 — SFI CALCULATION
# SFIi = Sum_j Wj . Sij . p   (equation 2)
# (p = 1/n : embedded in the standardized scores in step 4)
# ============================================================
"""

Input  : - outputs/data_scored.csv    (04_indicator_scoring.py)
         - outputs/pca_weights_Wj.csv (03_pca_weights.py)
Output : - outputs/sfi_summary.csv (Site, LandUse, Depth_class, SFI, SFI_method)
         - outputs/data_sfi.csv    (full dataset + SFI, SFI_method)
"""

from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("outputs")
SCORED_DATA_FILE = OUTPUT_DIR / "data_scored.csv"
WEIGHTS_FILE = OUTPUT_DIR / "pca_weights_Wj.csv"

SFI_SUMMARY_FILE = OUTPUT_DIR / "sfi_summary.csv"
SFI_FULL_FILE = OUTPUT_DIR / "data_sfi.csv"

# The 6 MSFI indicators, ALL required (no "without K" variant)
VARS_MSFI = ["pH_score", "CEC_score", "texture_score", "K_score", "P_score", "C_score"]

# Keep in sync with VAR_TO_SCORE_NAME in 03_pca_weights.py
VAR_TO_SCORE_NAME = {
    "pH": "pH_score",
    "CEC": "CEC_score",
    "clay": "texture_score",
    "K": "K_score",
    "P": "P_score",
    "C": "C_score",
}

SUMMARY_COLUMNS = ["Site", "LandUse", "Depth_class", "SFI", "SFI_method"]


# ============================================================
# FUNCTIONS
# ============================================================

def load_scored_data(path: Path) -> pd.DataFrame:
    """Load the scored data produced by 04_indicator_scoring.py."""
    if not path.exists():
        raise FileNotFoundError(
            f"File not found -> {path}. Did you run 04_indicator_scoring.py?"
        )
    return pd.read_csv(path)


def load_weights(path: Path, var_to_score: dict) -> pd.Series:
    """
    Load the Wj_normalized weights produced by 03_pca_weights.py and
    rename them to score column names (see VAR_TO_SCORE_NAME).
    """
    if not path.exists():
        raise FileNotFoundError(
            f"File not found -> {path}. Did you run 03_pca_weights.py?"
        )
    weights_table = pd.read_csv(path, index_col=0)
    return weights_table["Wj_normalized"].rename(index=var_to_score)


def align_weights(weights_scores: pd.Series, vars_msfi: list) -> pd.Series:
    """Reorder the weights to match vars_msfi exactly, and validate none are missing."""
    weights_aligned = weights_scores.reindex(vars_msfi)
    missing = weights_aligned[weights_aligned.isna()].index.tolist()
    if missing:
        raise ValueError(
            f"Missing weights after alignment for: {missing}. "
            f"weights_scores index: {weights_scores.index.tolist()}"
        )
    return weights_aligned


def compute_sfi(df: pd.DataFrame, vars_msfi: list, weights_aligned: pd.Series) -> pd.DataFrame:
    """
    Compute SFIi = Sum_j (Wj . Sij_standardized). p=1/n is already
    embedded in the scores (step 4) — do NOT multiply by p again here.
    """
    has_all = df[vars_msfi].notna().all(axis=1)

    df["SFI"] = np.nan
    df["SFI_method"] = "insufficient"

    df.loc[has_all, "SFI"] = (
        df.loc[has_all, vars_msfi].mul(weights_aligned, axis=1).sum(axis=1)
    )
    df.loc[has_all, "SFI_method"] = "complete_6_indicators"
    return df


def report_results(df: pd.DataFrame, weights_aligned: pd.Series, vars_msfi: list) -> None:
    """Print consistency checks (weights, scores, distribution, SFI)."""
    print("=== Aligned Wj weights (should sum to ~1) ===")
    print(weights_aligned)
    print("Sum of Wj weights:", weights_aligned.sum())

    print("\n=== Standardized scores Sij (before weighting) ===")
    print(df[vars_msfi].describe())

    print("\n=== Distribution of calculation methods ===")
    print(df["SFI_method"].value_counts(dropna=False))

    print("\n=== SFI summary ===")
    print(df["SFI"].describe())


def save_outputs(df: pd.DataFrame, summary_cols: list, summary_path: Path, full_path: Path) -> None:
    """Save a summary export and the full dataset, as two distinct files."""
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    df[summary_cols].to_csv(summary_path, index=False)
    df.to_csv(full_path, index=False)
    print(f"\nSaved SFI summary -> {summary_path}")
    print(f"Saved full dataset -> {full_path}")


def run_sfi_calculation(
    scored_data_file: Path = SCORED_DATA_FILE,
    weights_file: Path = WEIGHTS_FILE,
    vars_msfi: list = VARS_MSFI,
    var_to_score: dict = VAR_TO_SCORE_NAME,
    summary_cols: list = SUMMARY_COLUMNS,
    summary_path: Path = SFI_SUMMARY_FILE,
    full_path: Path = SFI_FULL_FILE,
) -> pd.DataFrame:
    """Full pipeline for step 5."""
    data = load_scored_data(scored_data_file)

    missing_cols = [v for v in vars_msfi if v not in data.columns]
    if missing_cols:
        raise KeyError(f"Score columns missing from `data`: {missing_cols}. Check 04_indicator_scoring.py.")

    weights_scores = load_weights(weights_file, var_to_score)
    weights_aligned = align_weights(weights_scores, vars_msfi)

    data = compute_sfi(data, vars_msfi, weights_aligned)
    report_results(data, weights_aligned, vars_msfi)

    missing_summary_cols = [c for c in summary_cols if c not in data.columns]
    if missing_summary_cols:
        raise KeyError(f"Expected summary columns missing: {missing_summary_cols}.")

    save_outputs(data, summary_cols, summary_path, full_path)
    return data


# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    data = run_sfi_calculation()


=== Aligned Wj weights (should sum to ~1) ===
pH_score         0.146666
CEC_score        0.201414
texture_score    0.162115
K_score          0.164844
P_score          0.155328
C_score          0.169633
Name: Wj_normalized, dtype: float64
Sum of Wj weights: 0.9999999999999999

=== Standardized scores Sij (before weighting) ===
          pH_score    CEC_score  texture_score      K_score      P_score  \
count  1529.000000  1529.000000    1529.000000  1529.000000  1529.000000   
mean      0.613080     0.549640       0.640942     0.281622     0.218967   
std       0.230493     0.175163       0.239815     0.104021     0.080777   
min       0.200000     0.200000       0.200000     0.200000     0.200000   
25%       0.400000     0.400000       0.600000     0.200000     0.200000   
50%       0.600000     0.600000       0.600000     0.200000     0.200000   
75%       0.800000     0.600000       0.800000     0.400000     0.200000   
max       1.000000     1.000000       1.000000     0.800000     

In [4]:
# ============================================================
# STEP 6 — FINAL SFI CLASSIFICATION (Table 5, Bagherzadeh et al., 2018)
# ============================================================
"""

Input  : outputs/data_sfi.csv, produced by 05_sfi_calculation.py
Output : - outputs/data_sfi_classified.csv (full dataset + SFI_class)
         - outputs/sfi_final_summary.csv   (Site, LandUse, Depth_class,
           SFI, SFI_method, SFI_class — ready for Kruskal-Wallis, Dunn,
           figures, etc.)
"""

from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("outputs")
SFI_DATA_FILE = OUTPUT_DIR / "data_sfi.csv"

CLASSIFIED_FULL_FILE = OUTPUT_DIR / "data_sfi_classified.csv"
FINAL_SUMMARY_FILE = OUTPUT_DIR / "sfi_final_summary.csv"

# Table 5, Bagherzadeh et al., 2018
CLASS_LABELS = ["Very low", "Low", "Moderate", "High", "Very high"]
CLASS_BREAKS = [-np.inf, 0.25, 0.50, 0.75, 0.90, np.inf]

SUMMARY_COLUMNS = ["Site", "LandUse", "Depth_class", "SFI", "SFI_method", "SFI_class"]


# ============================================================
# FUNCTIONS
# ============================================================

def load_sfi_data(path: Path) -> pd.DataFrame:
    """Load the dataset with the SFI already calculated, from 05_sfi_calculation.py."""
    if not path.exists():
        raise FileNotFoundError(
            f"File not found -> {path}. Did you run 05_sfi_calculation.py?"
        )
    return pd.read_csv(path)


def classify_sfi(df: pd.DataFrame, breaks: list, labels: list) -> pd.DataFrame:
    """
    Classify the SFI into 5 levels (Table 5). No re-division by 5: the
    SFI is already on the expected scale (see module docstring). pd.cut
    returns NaN for missing SFI values ("insufficient" observations from
    step 5), which is the intended behaviour.
    """
    df["SFI_class"] = pd.cut(df["SFI"], bins=breaks, labels=labels)
    return df


def check_classification_consistency(df: pd.DataFrame) -> None:
    """Check that SFI_class is missing exactly where SFI_method == 'insufficient'."""
    na_class = df["SFI_class"].isna()
    na_sfi = df["SFI"].isna()
    if not (na_class == na_sfi).all():
        mismatched = (na_class != na_sfi).sum()
        print(f"WARNING: SFI_class / SFI mismatch on {mismatched} observation(s) — investigate.")
    else:
        print("OK: SFI_class is consistent with missing SFI values.")


def report_results(df: pd.DataFrame) -> None:
    """Print the class distribution and the observed SFI range."""
    print("=== SFI_class distribution (Table 5, Bagherzadeh et al., 2018) ===")
    print(df["SFI_class"].value_counts(dropna=False))

    print("\n=== Observed SFI min/max (expected within [0.2, 1.0], see docstring) ===")
    print(df["SFI"].agg(["min", "max"]))


def save_outputs(df: pd.DataFrame, summary_cols: list, full_path: Path, summary_path: Path) -> None:
    """Save the full classified dataset and a summary, in outputs/, with no duplicate content."""
    full_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(full_path, index=False)
    df[summary_cols].to_csv(summary_path, index=False)
    print(f"\nSaved full classified dataset -> {full_path}")
    print(f"Saved final summary -> {summary_path}")


def run_classification(
    sfi_data_file: Path = SFI_DATA_FILE,
    breaks: list = CLASS_BREAKS,
    labels: list = CLASS_LABELS,
    summary_cols: list = SUMMARY_COLUMNS,
    full_path: Path = CLASSIFIED_FULL_FILE,
    summary_path: Path = FINAL_SUMMARY_FILE,
) -> pd.DataFrame:
    """Full pipeline for step 6."""
    data = load_sfi_data(sfi_data_file)

    if "SFI" not in data.columns:
        raise KeyError("Column 'SFI' is missing from `data`. Check 05_sfi_calculation.py.")

    data = classify_sfi(data, breaks, labels)
    check_classification_consistency(data)
    report_results(data)

    missing_summary_cols = [c for c in summary_cols if c not in data.columns]
    if missing_summary_cols:
        raise KeyError(f"Expected final summary columns missing: {missing_summary_cols}.")

    save_outputs(data, summary_cols, full_path, summary_path)
    return data


# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    data = run_classification()


OK: SFI_class is consistent with missing SFI values.
=== SFI_class distribution (Table 5, Bagherzadeh et al., 2018) ===
SFI_class
Low          983
Moderate     533
Very low      13
High           0
Very high      0
Name: count, dtype: int64

=== Observed SFI min/max (expected within [0.2, 1.0], see docstring) ===
min    0.200000
max    0.740934
Name: SFI, dtype: float64

Saved full classified dataset -> outputs\data_sfi_classified.csv
Saved final summary -> outputs\sfi_final_summary.csv


In [5]:
# ============================================================
# SUPPLEMENTARY SCRIPT — PCA SUMMARY TABLE + SFI CORRELATION CIRCLE
# (not part of the 01-06 pipeline; run after 05_sfi_calculation.py)
# ============================================================
"""
Purpose
-------
Produce a manuscript-ready PCA summary table (eigenvalues, variance
proportions, variable loadings) for the 6 MSFI properties, and a
correlation circle showing where SFI projects relative to those
properties. SFI is added as a supplementary quantitative variable: its
position is the Pearson correlation between SFI and each principal
component's individual score, which does not affect the PCA itself.

Input  : outputs/data_sfi.csv, produced by 05_sfi_calculation.py
Output : - outputs/pca_summary_table.csv
         - outputs/pca_summary_table.png
         - outputs/pca_sfi_correlation_circle.png
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("outputs")
SFI_DATA_FILE = OUTPUT_DIR / "data_sfi.csv"

SUMMARY_TABLE_CSV = OUTPUT_DIR / "pca_summary_table.csv"
SUMMARY_TABLE_PNG = OUTPUT_DIR / "pca_summary_table.png"
BIPLOT_PNG = OUTPUT_DIR / "pca_sfi_correlation_circle.png"

PCA_VARIABLES = ["pH", "CEC", "clay", "K", "P", "C"]
DISPLAY_NAMES = {
    "pH": "pH", "CEC": "CEC", "clay": "Clay",
    "K": "K content", "P": "P content", "C": "C content",
}

N_DIMS_DISPLAYED = 2  # PC1 / PC2, matching the correlation circle


# ============================================================
# FUNCTIONS
# ============================================================

def load_sfi_data(path: Path) -> pd.DataFrame:
    """Load the dataset with SFI, produced by 05_sfi_calculation.py."""
    if not path.exists():
        raise FileNotFoundError(f"File not found -> {path}. Did you run 05_sfi_calculation.py?")
    return pd.read_csv(path)


def compute_pca(df: pd.DataFrame, variables: list) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Fit PCA on the MSFI properties; return eigenvalue table, variable coordinates, individual scores."""
    pca_data = df[variables].dropna()
    X_scaled = StandardScaler().fit_transform(pca_data)

    pca = PCA()
    scores = pca.fit_transform(X_scaled)

    eigenvalues = pca.explained_variance_
    variance_pct = pca.explained_variance_ratio_ * 100
    cum_variance_pct = np.cumsum(variance_pct)
    dims = [f"PC{i + 1}" for i in range(len(eigenvalues))]

    eig_table = pd.DataFrame(
        {"eigenvalue": eigenvalues, "proportion": variance_pct, "cumulative": cum_variance_pct},
        index=dims,
    )

    loadings = pca.components_.T
    var_coord_df = pd.DataFrame(loadings * np.sqrt(eigenvalues), index=variables, columns=dims)

    scores_df = pd.DataFrame(scores, index=pca_data.index, columns=dims)
    return eig_table, var_coord_df, scores_df


def compute_sfi_supplementary_coords(df: pd.DataFrame, scores_df: pd.DataFrame, dims: list) -> pd.Series:
    """
    Project SFI as a supplementary quantitative variable: its coordinate
    on a component equals its Pearson correlation with that component's
    individual score. Does not influence the PCA itself.
    """
    sfi = df.loc[scores_df.index, "SFI"]
    valid = sfi.notna()
    coords = {dim: pearsonr(sfi[valid], scores_df.loc[valid, dim])[0] for dim in dims}
    return pd.Series(coords, name="SFI")


def build_summary_table(eig_table: pd.DataFrame, var_coord_df: pd.DataFrame, display_names: dict, n_dims: int) -> pd.DataFrame:
    """Assemble a single table: eigenvalue/proportion/cumulative rows, then variable loadings."""
    dims = eig_table.index[:n_dims]
    stats_block = eig_table.loc[dims, ["eigenvalue", "proportion", "cumulative"]].T
    stats_block.index = ["Eigenvalue", "Proportion", "Cumulative"]

    loadings_block = var_coord_df[dims].rename(index=display_names)
    loadings_block.index.name = "Variable"

    return stats_block, loadings_block


def save_summary_csv(stats_block: pd.DataFrame, loadings_block: pd.DataFrame, path: Path) -> None:
    """Save both blocks to a single CSV, stacked with a blank separator line."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        stats_block.to_csv(f)
        f.write("\n")
        loadings_block.to_csv(f)
    print(f"Saved PCA summary table (CSV) -> {path}")


def render_summary_png(stats_block: pd.DataFrame, loadings_block: pd.DataFrame, path: Path) -> None:
    """Render the summary table as a PNG image, bolding each variable's dominant loading."""
    n_rows = len(stats_block) + len(loadings_block) + 1  # +1 for header row
    fig, ax = plt.subplots(figsize=(5, 0.45 * n_rows + 1))
    ax.axis("off")

    dims = list(stats_block.columns)
    col_labels = [""] + dims
    stats_rows = [[idx] + [f"{v:.3f}" for v in row] for idx, row in zip(stats_block.index, stats_block.values)]
    loadings_rows = [[idx] + [f"{v:.3f}" for v in row] for idx, row in zip(loadings_block.index, loadings_block.values)]

    table_data = stats_rows + [[""] * len(col_labels)] + loadings_rows
    table = ax.table(cellText=table_data, colLabels=col_labels, loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.4)

    # Bold the dominant (max |loading|) component for each variable row
    n_stats_rows = len(stats_rows) + 1  # +1 for blank separator row
    for i, (_, row) in enumerate(loadings_block.iterrows()):
        dominant_col = int(np.argmax(np.abs(row.values))) + 1  # +1 to skip the "Variable" column
        cell = table[(n_stats_rows + i + 1, dominant_col)]  # +1 for header row
        cell.set_text_props(fontweight="bold")

    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved PCA summary table (PNG) -> {path}")


def plot_sfi_correlation_circle(var_coord_df: pd.DataFrame, sfi_coords: pd.Series, eig_table: pd.DataFrame, display_names: dict, path: Path) -> None:
    """Correlation circle (PC1 x PC2): MSFI properties as solid arrows, SFI as a dashed supplementary arrow."""
    pc1_pct = eig_table.loc["PC1", "proportion"]
    pc2_pct = eig_table.loc["PC2", "proportion"]

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    circle = plt.Circle((0, 0), 1, fill=False, color="grey", linestyle="--")
    ax.add_patch(circle)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.axvline(0, color="grey", linewidth=0.8)

    for var in var_coord_df.index:
        x, y = var_coord_df.loc[var, "PC1"], var_coord_df.loc[var, "PC2"]
        ax.annotate("", xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.5))
        ax.text(x * 1.1, y * 1.1, display_names.get(var, var), ha="center", va="center", fontsize=10, color="steelblue")

    x_sfi, y_sfi = sfi_coords["PC1"], sfi_coords["PC2"]
    ax.annotate("", xy=(x_sfi, y_sfi), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="firebrick", lw=2, linestyle="--"))
    ax.text(x_sfi * 1.1, y_sfi * 1.1, "SFI", ha="center", va="center", fontsize=11, color="firebrick", fontweight="bold")

    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.set_xlabel(f"PC1 ({pc1_pct:.1f}%)")
    ax.set_ylabel(f"PC2 ({pc2_pct:.1f}%)")
    ax.set_title("PCA correlation circle — MSFI properties and SFI (supplementary)")

    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    print(f"Saved SFI correlation circle -> {path}")


def run_pca_summary_and_biplot(
    sfi_data_file: Path = SFI_DATA_FILE,
    variables: list = PCA_VARIABLES,
    display_names: dict = DISPLAY_NAMES,
    n_dims: int = N_DIMS_DISPLAYED,
    summary_csv: Path = SUMMARY_TABLE_CSV,
    summary_png: Path = SUMMARY_TABLE_PNG,
    biplot_png: Path = BIPLOT_PNG,
) -> None:
    """Full pipeline for this supplementary script."""
    data = load_sfi_data(sfi_data_file)

    missing = [v for v in variables + ["SFI"] if v not in data.columns]
    if missing:
        raise KeyError(f"Columns missing from `data`: {missing}. Check 05_sfi_calculation.py.")

    eig_table, var_coord_df, scores_df = compute_pca(data, variables)
    sfi_coords = compute_sfi_supplementary_coords(data, scores_df, list(eig_table.index[:2]))

    stats_block, loadings_block = build_summary_table(eig_table, var_coord_df, display_names, n_dims)
    print("=== PCA summary (eigenvalues / proportion / cumulative, %) ===")
    print(stats_block)
    print("\n=== Variable loadings ===")
    print(loadings_block)
    print("\n=== SFI supplementary coordinates ===")
    print(sfi_coords)

    save_summary_csv(stats_block, loadings_block, summary_csv)
    render_summary_png(stats_block, loadings_block, summary_png)
    plot_sfi_correlation_circle(var_coord_df, sfi_coords, eig_table, display_names, biplot_png)


# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    run_pca_summary_and_biplot()


=== PCA summary (eigenvalues / proportion / cumulative, %) ===
                  PC1        PC2
Eigenvalue   1.938504   1.652169
Proportion  32.287276  27.518141
Cumulative  32.287276  59.805417

=== Variable loadings ===
                PC1       PC2
Variable                     
pH        -0.042699 -0.672763
CEC        0.787427 -0.295009
Clay       0.633787 -0.547382
K content  0.644458  0.015473
P content  0.244581  0.712496
C content  0.663181  0.552278

=== SFI supplementary coordinates ===
PC1    0.884293
PC2   -0.199923
Name: SFI, dtype: float64
Saved PCA summary table (CSV) -> outputs\pca_summary_table.csv
Saved PCA summary table (PNG) -> outputs\pca_summary_table.png
Saved SFI correlation circle -> outputs\pca_sfi_correlation_circle.png
